In [1]:
df = spark.sql("SELECT * FROM supply_guard.dbo.Silver_inventory LIMIT 1000")
display(df)

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8656ae84-8cac-441a-be41-fa881a09c451)

In [4]:
from pyspark.sql.functions import col

inventory_df = spark.read.table("supply_guard.dbo.Silver_inventory")

gold_inventory = inventory_df.withColumn(
    "InventoryValue",
    col("onHand") * col("Price")
)

gold_inventory.write.mode("overwrite").saveAsTable(
    "supply_guard.dbo.gold_inventory_optimization"
)

print("Gold Inventory Created")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 6, Finished, Available, Finished, False)

Gold Inventory Created


In [5]:
vendor_df = spark.read.table(
    "supply_guard.dbo.silver_vendor_sales_summary"
)

gold_vendor = vendor_df.select(
    "VendorNumber",
    "VendorName",
    "Brand",
    "GrossProfit",
    "ProfitMargin",
    "StockTurnover",
    "SalestoPurchaseRatio",
    "TotalSalesDollars",
    "TotalPurchaseDollars"
)

gold_vendor.write.mode("overwrite").saveAsTable(
    "supply_guard.dbo.gold_vendor_performance"
)

print("Gold Vendor Performance Created")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 7, Finished, Available, Finished, False)

Gold Vendor Performance Created


In [17]:
from pyspark.sql.functions import sum, countDistinct

# Load tables
purchase_df = spark.read.table("supply_guard.dbo.silver_purchase")
invoice_df = spark.read.table("supply_guard.dbo.silver_vendor_invoice")

# Purchase Summary
purchase_summary = purchase_df.groupBy(
    "VendorNumber",
    "VendorName"
).agg(
    sum("Dollars").alias("TotalPurchaseSpend"),
    sum("Quantity").alias("TotalPurchaseQuantity"),
    countDistinct("PONumber").alias("TotalPOs")
)

# Invoice Summary
invoice_summary = invoice_df.groupBy(
    "VendorNumber"
).agg(
    sum("Freight").alias("TotalFreight"),
    countDistinct("InvoiceDate").alias("InvoiceCount")
)

# Join Aggregated Tables
gold_procurement = purchase_summary.join(
    invoice_summary,
    "VendorNumber",
    "left"
)

# Save Gold Table
gold_procurement.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("supply_guard.dbo.gold_procurement_intelligence")

print("Gold Procurement Created")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 19, Finished, Available, Finished, False)

Gold Procurement Created


In [21]:
vendor_dim = spark.read.table(
    "supply_guard.dbo.silver_vendor_sales_summary"
).select(
    "VendorNumber",
    "VendorName"
).dropDuplicates()

vendor_dim.write \
    .mode("overwrite") \
    .saveAsTable("supply_guard.dbo.dim_vendor")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 23, Finished, Available, Finished, False)

In [22]:
product_dim = spark.read.table(
    "supply_guard.dbo.silver_purchase_prices"
).select(
    "Brand",
    "Description",
    "Size",
    "Classification"
).dropDuplicates()

product_dim.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("supply_guard.dbo.dim_product")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 24, Finished, Available, Finished, False)

In [23]:
store_dim = spark.read.table(
    "supply_guard.dbo.silver_inventory"
).select(
    "Store",
    "City"
).dropDuplicates()

store_dim.write \
    .mode("overwrite") \
    .saveAsTable("supply_guard.dbo.dim_store")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 25, Finished, Available, Finished, False)

In [24]:
from pyspark.sql.functions import explode, sequence, to_date, lit

date_dim = spark.sql("""
SELECT explode(
    sequence(
        to_date('2015-01-01'),
        to_date('2025-12-31'),
        interval 1 day
    )
) as Date
""")

date_dim.createOrReplaceTempView("date_dim")

date_dim = spark.sql("""
SELECT
    Date,
    year(Date) as Year,
    quarter(Date) as Quarter,
    month(Date) as Month,
    day(Date) as Day
FROM date_dim
""")

date_dim.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("supply_guard.dbo.dim_date")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 26, Finished, Available, Finished, False)

In [25]:
from pyspark.sql.functions import col

fact_inventory = spark.read.table(
    "supply_guard.dbo.silver_inventory"
).withColumn(
    "InventoryValue",
    col("onHand") * col("Price")
)

fact_inventory.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("supply_guard.dbo.fact_inventory")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 27, Finished, Available, Finished, False)

In [26]:
fact_vendor = spark.read.table(
    "supply_guard.dbo.silver_vendor_sales_summary"
)

fact_vendor.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("supply_guard.dbo.fact_vendor_performance")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 28, Finished, Available, Finished, False)

In [27]:
from pyspark.sql.functions import sum,countDistinct

purchase_df = spark.read.table(
    "supply_guard.dbo.silver_purchase"
)

invoice_df = spark.read.table(
    "supply_guard.dbo.silver_vendor_invoice"
)

purchase_summary = purchase_df.groupBy(
    "VendorNumber",
    "VendorName"
).agg(
    sum("Dollars").alias("TotalPurchaseSpend"),
    sum("Quantity").alias("TotalPurchaseQuantity"),
    countDistinct("PONumber").alias("TotalPOs")
)

invoice_summary = invoice_df.groupBy(
    "VendorNumber"
).agg(
    sum("Freight").alias("TotalFreight"),
    countDistinct("InvoiceDate").alias("InvoiceCount")
)

fact_procurement = purchase_summary.join(
    invoice_summary,
    "VendorNumber",
    "left"
)

fact_procurement.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("supply_guard.dbo.fact_procurement")

StatementMeta(, 88f66c13-4844-4aac-b6ad-d1969389f8ee, 29, Finished, Available, Finished, False)

In [1]:
# Read source tables
inventory = spark.read.table("supply_guard.dbo.Silver_inventory")
purchase = spark.read.table("supply_guard.dbo.Silver_purchase")

# Delete old tables if exist
spark.sql("DROP TABLE IF EXISTS supply_guard.dbo.dim_store")
spark.sql("DROP TABLE IF EXISTS supply_guard.dbo.dim_vendor")

# Create clean dim_store
dim_store = (
    inventory
    .select("Store", "City")
    .filter("City IS NOT NULL")
    .dropDuplicates(["Store"])
)

# Create clean dim_vendor
dim_vendor = (
    purchase
    .select("VendorNumber", "VendorName")
    .dropDuplicates(["VendorNumber"])
)

# Save new dimensions
dim_store.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("supply_guard.dbo.dim_store")

dim_vendor.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("supply_guard.dbo.dim_vendor")

print("dim_store and dim_vendor recreated successfully")

StatementMeta(, 6fc374d6-d779-47e9-b502-ad21fa9dae4c, 3, Finished, Available, Finished, False)

dim_store and dim_vendor recreated successfully
